# Anime

In [1]:
import pandas as pd
import numpy as np
import plotly.express as px

In [2]:
import pandas as pd
from matplotlib import pyplot as plt
import seaborn as sns
import numpy as np
import plotly.graph_objects as go
import ast
import plotly.express as px

In [3]:
# load dataset
df_profiles = pd.read_csv("../../data/cleaned/profiles_cleaned.csv")
df_person_anime_works = pd.read_csv("../../data/cleaned/person_anime_works_cleaned.csv")
df_person_details = pd.read_csv("../../data/cleaned/person_details_cleaned.csv")
df_person_alternate_names = pd.read_csv("../../data/cleaned/person_alternate_names_cleaned.csv")



In [4]:
df_person_details["birthday"] = pd.to_datetime(df_person_details["birthday"], errors="coerce")
df_person_details["birth_year"] = df_person_details["birthday"].dt.year.astype("Int64")

df_person_details["favorites"] = pd.to_numeric(df_person_details["favorites"], errors="coerce")


In [5]:
#Profiles
print("Righe totali profiles:", len(df_profiles))
print("Utenti unici (username):", df_profiles["username"].nunique())
print("Location non-missing:", df_profiles["location"].notna().mean().round(3), "→ quota")
print("Joined non-missing:", df_profiles["joined"].notna().mean().round(3), "→ quota")

profiles = df_profiles

Righe totali profiles: 337154
Utenti unici (username): 337154
Location non-missing: 1.0 → quota
Joined non-missing: 0.995 → quota


## In quale aree del mondo gli anime sono maggiormente seguiti?


In [6]:
# per errore che mi dava sotto
import warnings
warnings.filterwarnings("ignore", category=DeprecationWarning)

df = df_profiles.copy()

# extract country from location
profiles["country"] = (
    profiles["location"]
    .astype("string")
    .str.strip()
    .replace({"": pd.NA})
    .str.split(",")
    .str[-1]
    .str.strip()
)

#country names
country_fix = {
    "Usa": "United States",
    "US": "United States",
    "United States Of America": "United States",
    "UK": "United Kingdom",
    "England": "United Kingdom",
    "Scotland": "United Kingdom",
    "Russian Federation": "Russia",
    "Russia": "Russia",
    "South Korea": "South Korea",
    "Korea, Republic of": "South Korea",
    "Korea, Rep.": "South Korea",

}
profiles["country"] = profiles["country"].replace(country_fix)

profiles["country"] = profiles["country"].where(
    profiles["country"].isna(),
    profiles["country"].str.replace(r"\s+", " ", regex=True).str.title()
)

#controlli country
print("Righe totali:", len(profiles))
print("Utenti unici:", profiles["username"].nunique())

loc_ok = profiles["location"].notna().mean()
print("Quota location non-missing:", round(loc_ok, 3))

country_ok = profiles["country"].notna().mean()
print("Quota country estratto non-missing:", round(country_ok, 3))

print("Paesi unici estratti:", profiles["country"].dropna().nunique())
print("Top 10 paesi (utenti unici):")
print(
    profiles.dropna(subset=["country"])
            .groupby("country")["username"].nunique()
            .sort_values(ascending=False)
            .head(10)
)
#unique users per country
country_counts = (
    profiles.dropna(subset=["country"])
            .groupby("country")["username"]
            .nunique()
            .reset_index(name="n_users")
)

#which countries are not recognised by Plotly?
valid_countries = px.data.gapminder()["country"].unique()
not_recognised = (
    country_counts.loc[~country_counts["country"].isin(valid_countries), ["country", "n_users"]]
    .sort_values("n_users", ascending=False)
)
not_recognised.head(30)

# log scale
country_counts["log_users"] = np.log10(country_counts["n_users"] + 1)

# choropleth map
fig = px.choropleth(
    country_counts,
    locations="country",
    locationmode="country names",
    color="log_users",
    hover_name="country",
    hover_data={"n_users": True, "log_users": ":.2f"},
    title="Dove gli anime sono maggiormente seguiti? (utenti per paese, scala log10)",
)

fig.update_layout(
    coloraxis_colorbar=dict(title="log10(utenti)", tickformat=".1f")
)

fig.update_geos(showland=True, landcolor="lightgrey", showocean=True, oceancolor="#e6f2ff")

#Layout
fig.update_layout(template="plotly_white", width=1100, height=600)
fig.show()


Righe totali: 337154
Utenti unici: 337154
Quota location non-missing: 1.0
Quota country estratto non-missing: 1.0
Paesi unici estratti: 22
Top 10 paesi (utenti unici):
country
Japan             98316
United States     65240
Germany           26348
United Kingdom    19520
Thailand          16416
Argentina         13230
China             13196
Spain             12916
France            10041
Australia          9691
Name: username, dtype: int64


La mappa evidenzia paesi in cui il dataset registra una maggiore presenza di utenti appassionati di anime. L'intensità del colore cresce all'aumentare del numero di utenti unici per paese; l'uso della trasformazione log(n_users + 1) comprime le differenze estreme di valore e rende visibili anche paesi con valori più bassi.

#### A questo punto possiamo vedere come l'interesse per gli anime si è sviluppato durante il corso del tempo, considerando i paesi dove esse son maggiormente diffuse.

## Come si è evoluta nel tempo la crescita della community/nuovi utenti per gli anime nelle aree dove sono più concentrate?

In [7]:
df = df_profiles.copy()

# converto joined + extract year
profiles["joined"] = pd.to_datetime(profiles["joined"], errors="coerce")
profiles["joined_year"] = profiles["joined"].dt.year

#filter year
profiles = profiles[(profiles["joined_year"] >= 1917) & (profiles["joined_year"] <= 2024)]

#Top 6 countries
top_countries = (
    profiles.dropna(subset=["country"])
            .groupby("country")["username"]
            .nunique()
            .sort_values(ascending=False)
            .head(6)
            .index
)
profiles_top = profiles[profiles["country"].isin(top_countries)].copy()
# new users to year
trend = (
    profiles_top.dropna(subset=["joined_year", "country"])
                .groupby(["joined_year", "country"])["username"]
                .nunique()
                .reset_index(name="n_users")
)

fig = px.line(trend, x="joined_year", y="n_users", color="country", markers=True)

#line chart
fig = px.line(
    trend,
    x="joined_year",
    y="n_users",
    color="country",
    markers=True,
    title="Crescita della community nel tempo: nuovi utenti registrati per anno (top paesi)"
)

fig.update_layout(
    template="plotly_white",
    width=1100, height=600,
    xaxis_title="Anno",
    yaxis_title="Nuovi utenti" ,
    legend_title_text = "Paese"
)

fig.show()


Il grafico mostra l'evoluzione nel tempo dei nuovi utenti registrati ogni anno nei 6 paesi con più utenti nel dataset;


#### E l'evoluzione a livello globale? Vediamo come si modifica a livello temporale ...

## Crescita della community nel tempo: nuovi utenti registrati per anno

In [8]:
df = df_profiles.copy()
#convert joined to datetime and extract  year
df["joined"] = pd.to_datetime(df["joined"], errors="coerce")
df["joined_year"] = df["joined"].dt.year

users_per_year = (
    df.dropna(subset=["joined_year"])
      .assign(joined_year=lambda x: x["joined_year"].astype(int))
      .groupby("joined_year", observed=True)["username"]
      .nunique()
      .reset_index(name="new_users")
      .sort_values("joined_year")
)

users_per_year = users_per_year[users_per_year["joined_year"].between(2005, 2024)].copy()

#line chart
fig = px.line(
    users_per_year,
    x="joined_year",
    y="new_users",
    markers=True,
    title="Crescita della community nel tempo: nuovi utenti registrati per anno",
    labels={"joined_year": "Anno", "new_users": "Nuovi utenti"},
    color_discrete_sequence=["#1f4e79"]
)
fig.update_layout(template="plotly_white", width=1100, height=600)
fig.show()

Il grafico mostra l’andamento dei nuovi utenti registrati ogni anno: si osserva una crescita nel tempo, con picchi marcati negli anni più recenti.

Dopo aver osservato la crescita della community nel tempo, analizziamo come sono composte le liste degli utenti, distinguendo  l'anno di registrazione, per capire se le abitudini di consumo variano tra utenti ‘vecchi’ e ‘nuovi’.

## Composizione delle liste per anno di iscrizione:

In [9]:
df = df_profiles.copy()
# anno iscrizione
df["joined"] = pd.to_datetime(df["joined"], errors="coerce")
df["joined_year"] = df["joined"].dt.year

# colonne di consumo
cols = ["watching", "completed", "on_hold", "dropped", "plan_to_watch"]
df[cols] = df[cols].apply(pd.to_numeric, errors="coerce").fillna(0)

# controlli
print("Righe totali:", len(df))
print("Quota joined_year non-missing:", round(df["joined_year"].notna().mean(), 3))

# aggregazione per anno
consumption_time = (
    df.dropna(subset=["joined_year"])
      .assign(joined_year=lambda x: x["joined_year"].astype(int))
      .groupby("joined_year", observed=True)[cols]
      .sum()
      .reset_index()
      .sort_values("joined_year")
)
#periodo
consumption_time = consumption_time[consumption_time["joined_year"].between(2005, 2024)].copy()

#trasformo in percentuale, così da avere dati più omogenei
den = consumption_time[cols].sum(axis=1)
consumption_pct = consumption_time.copy()
consumption_pct[cols] = consumption_pct[cols].div(den, axis=0) * 100

#grafico
consumption_long = consumption_pct.melt(
    id_vars="joined_year",
    value_vars=cols,
    var_name="status",
    value_name="share"
)

fig = px.area(
    consumption_long,
    x="joined_year",
    y="share",
    color="status",
    title="Composizione delle liste per coorte di iscrizione",
    labels={"joined_year": "Anno di iscrizione", "share": "Quota (%)", "status": "Status"}
)

fig.update_layout(template="plotly_white", width=1100, height=600)
fig.update_yaxes(ticksuffix="%", range=[0,100], dtick=10)
fig.update_traces(hovertemplate="Quota=%{y:.1f}%<br>Anno=%{x}<extra></extra>")
fig.show()

Righe totali: 337154
Quota joined_year non-missing: 0.995


Il grafico mostra, per ogni coorte di iscrizione (anno di registrazione), la composizione percentuale delle liste degli utenti nei diversi status (watching, completed, on_hold, dropped, plan_to_watch). Le quote sommano a 100% per anno e permettono di confrontare come cambia la struttura delle liste tra coorti.

### Aggiungerò un grafico verificando ogni anno quanti anime sono stati completati, per vedere come nel tempo gli utenti hanno completato gli anime

In [10]:
#person_anime_works
works = df_person_anime_works.copy()

print("Righe totali:", len(works))
print("Position missing:", works["position"].isna().mean().round(3), "→ quota")
print("Person ID unici:", works["person_mal_id"].nunique() if "person_mal_id" in works.columns else "N/A")
print("Anime ID unici:", works["anime_mal_id"].nunique() if "anime_mal_id" in works.columns else "N/A")
print("Ruoli unici (raw):", works["position"].nunique(dropna=True))

Righe totali: 458091
Position missing: 0.0 → quota
Person ID unici: 26236
Anime ID unici: 20148
Ruoli unici (raw): 80036


## Quali ruoli sono più diffusi nell'industria anime?

In [11]:
# normalizzazione ruoli (no per-episodio)
works = df_person_anime_works.copy()

works["position_norm"] = (
    works["position"]
    .astype(str)
    .str.replace(r"\s*\(ep\s*\d+\)\s*$", "", regex=True)
    .str.strip()
)
works["position_norm"] = (
    works["position"]
    .astype("string")
    .str.replace(r"\s*\(ep\s*\d+\)\s*$", "", regex=True)
    .str.replace(r"\s+", " ", regex=True)
    .str.strip()
)
works.loc[works["position_norm"].isin(["", "nan"]), "position_norm"] = pd.NA

#conteggio ruoli
role_counts = (
    works["position_norm"]
    .dropna()
    .value_counts()
    .reset_index()
)

role_counts.columns = ["role", "count"]

# top 10
top_roles = role_counts.head(10).sort_values("count", ascending=False).copy()
#percentuale sul tot del datset
total = role_counts["count"].sum()
top_roles["percentage"] = top_roles["count"] / total * 100

# grafico
fig = px.bar(
    top_roles,
    x="count",
    y="role",
    orientation="h",
    text=top_roles["percentage"].round(1).astype(str) + "%",
    title="Quali sono i ruoli più diffusi nell'industria degli anime?",
    color="role",
    color_discrete_sequence=px.colors.qualitative.Set3
)

fig.update_layout(
    xaxis_title="Numero di occorrenze",
    yaxis_title="Ruolo",
    showlegend=False,
    width=1000,
    height=600
)

fig.update_traces(textposition="outside")
fig.show()


Il grafico evidenzia i 10 ruoli più frequenti nell'industria anime, misurati con il numero di totale di occorrenze nel dataset.

Sorge spontaneo chiedersi da chi sono ricoperti tali ruoli? Da quale generazione?

## Quali sono le generazioni più diffuse nell'industria anime?

In [12]:
#person details
details = df_person_details.copy()

print("Righe totali:", len(details))
print("Person ID missing:", details["person_mal_id"].isna().mean().round(3), "→ quota")
print("Person ID unici:", details["person_mal_id"].nunique() if "person_mal_id" in details.columns else "N/A")

# campi chiave (se esistono)
print("Birthday missing:", details["birthday"].isna().mean().round(3), "→ quota" if "birthday" in details.columns else "Birthday: N/A")
print("Birth_year missing:", details["birth_year"].isna().mean().round(3), "→ quota" if "birth_year" in details.columns else "Birth_year: N/A")
print("Favorites missing:", details["favorites"].isna().mean().round(3), "→ quota" if "favorites" in details.columns else "Favorites: N/A")

# stringhe principali (se esistono)
print("Name missing:", details["name"].isna().mean().round(3), "→ quota" if "name" in details.columns else "Name: N/A")
print("Given_name missing:", details["given_name"].isna().mean().round(3), "→ quota" if "given_name" in details.columns else "Given_name: N/A")
print("Family_name missing:", details["family_name"].isna().mean().round(3), "→ quota" if "family_name" in details.columns else "Family_name: N/A")
print("Relevant_location missing:", details["relevant_location"].isna().mean().round(3), "→ quota" if "relevant_location" in details.columns else "Relevant_location: N/A")

Righe totali: 76594
Person ID missing: 0.0 → quota
Person ID unici: 76594
Birthday missing: 0.776 → quota
Birth_year missing: 0.776 → quota
Favorites missing: 0.0 → quota
Name missing: 0.0 → quota
Given_name missing: 0.394 → quota
Family_name missing: 0.243 → quota
Relevant_location missing: 0.0 → quota


In [13]:
# prendo person_mal_id
works_people = (
    df_person_anime_works[["person_mal_id"]]
    .copy()
)
#converto in numerico e tolgo righe mancanti o duplicati (controllo ch eho aggiunto per sicurezza)
works_people["person_mal_id"] = pd.to_numeric(works_people["person_mal_id"], errors="coerce").astype("Int64")
works_people = works_people.dropna(subset=["person_mal_id"]).drop_duplicates()

# recupero anni di nascita da details
details_birth = (
    df_person_details[["person_mal_id", "birthday"]]
    .copy()
)
#conversioni di birthday
details_birth["person_mal_id"] = pd.to_numeric(details_birth["person_mal_id"], errors="coerce").astype("Int64")
details_birth["birthday"] = pd.to_datetime(details_birth["birthday"], errors="coerce", utc=True)
details_birth["birth_year"] = details_birth["birthday"].dt.year.astype("Int64")

# filtro anni (prima di 1900 e dopo 2024 gli anni li considero 'non palusibili' per evitare dati strani)
details_birth.loc[~details_birth["birth_year"].between(1900, 2024), "birth_year"] = pd.NA

#aggiungo birth_year
staff = works_people.merge(details_birth[["person_mal_id", "birth_year"]], on="person_mal_id", how="left")
details_birth = details_birth.drop_duplicates(subset=["person_mal_id"]) #verifico duplicati (controllo in +)

print("People nei credits (unique person_mal_id):", staff["person_mal_id"].nunique())
print("People nei credits con birth_year disponibile:", staff["birth_year"].notna().sum())
print("Quota birth_year disponibile:", round(staff["birth_year"].notna().mean(), 3))

# generazioni
labels = ["Silent Generation", "Boomers", "Gen X", "Millennials", "Gen Z", "Gen Alpha"]
staff["generation"] = pd.cut(
    staff["birth_year"],
    bins=[-np.inf, 1945, 1964, 1980, 1996, 2012, 2024],
    labels=labels ,
    right=True
)

gen_counts = (
    staff.dropna(subset=["generation"])
         .groupby("generation", observed=True)["person_mal_id"]
         .nunique()
         .reset_index(name="count")
)

gen_counts["generation"] = pd.Categorical(gen_counts["generation"], categories=labels, ordered=True)
gen_counts = gen_counts.sort_values("generation")
gen_counts["share"] = gen_counts["count"] / gen_counts["count"].sum() * 100

color_map = {
    "Silent Generation": "#A0AEC0",
    "Boomers": "#F6AD55",
    "Gen X": "#63B3ED",
    "Millennials": "#68D391",
    "Gen Z": "#FC8181",
    "Gen Alpha": "#B794F4",
}

fig = px.bar(
    gen_counts,
    x="generation",
    y="count",
    color="generation",
    color_discrete_map=color_map,
    text=gen_counts["share"].round(1).astype(str) + "%",
    title="Quali generazioni sono più diffuse tra le persone che fanno parte dell’industria anime?",
    labels={"count": "Numero di persone", "generation": "Generazione"},
)
fig.update_layout(template="plotly_white", width=1100, height=600, showlegend=False)
fig.update_traces(textposition="outside")
fig.update_yaxes(rangemode="tozero")
fig.show()

People nei credits (unique person_mal_id): 26236
People nei credits con birth_year disponibile: 6409
Quota birth_year disponibile: 0.244


Il grafico mostra la distribuzione delle generazioni tra le persone presenti nello staff, ogni barra evidenzia il numero di persone uniche per generazione, con la percentuale calcolata sul sottoinsieme per cui l'anno di nascita è disponibile.